# Project Elsa benchmark — Portugal / Millennium + WiZink

Two datasets, both from the **raw Intrum CAT feed** — not PROD, which holds only 42% of
these collections. They join on `(PORTFOLIO_ID, LOAN_ID)` and cover the same 150,514 loans
across **12 batches and two sellers**. Every guard and measured figure lives in the two
`.sql` headers; this notebook loads, checks and exports them.

| | batches | loans | registration capital | collections |
|---|---|---|---|---|
| Millennium | 11 | 120,543 | 906,359,149.41 | 158,078,425.40 |
| WiZink | 1 | 29,971 | 122,891,559.16 | 43,146,173.13 |
| **Total** | **12** | **150,514** | **1,029,250,708.57** | **201,224,598.53** |

### `accounts` — one row per loan, 8 columns

| column | what it is |
|---|---|
| `SELLER_NAME` | `Millennium` or `Wizink`. **Split on this before quoting any rate** — see below |
| `PORTFOLIO_ID` | the batch ref, e.g. `PT050030000000` (PAM's `COMBINED_REF`) |
| `LOAN_ID` | raw Intrum `CASE_ID`. PROD's key is `4050-<LOAN_ID>` |
| `REGISTRATION_DATE` | when Intrum opened the case — **not** origination |
| `REGISTRATION_CAPITAL` | **principal** (`ORIGINAL_CAPITAL`) at registration. Complete on every loan in both books, and the figure that reconciles to PAM. **The only money column** |
| `DEFAULT_DATE` | PROD's back-filled default date. 100% filled on both sellers |
| `DEFAULT_DATE_SUSPECT_FLAG` | 1,876 pre-1990 dates (earliest 1978-01-01), flagged rather than nulled. All Millennium |
| `PRICING_SEGMENT` | `SME` if any corporate debtor, else `Individual`. **Millennium only** — WiZink's 5 SME loans are dropped by rule |

### `payments` — one row per (loan, payment date), 6 columns. All EUR.

798,774 rows over 67,503 loans that ever paid. EUR 201,224,598.53, of which
EUR 5,407,533.41 (2.69%) has no date and cannot enter a time series — and that undated
cash is **entirely Millennium**; WiZink's is a single EUR -1.62 reversal.

### What changed, 2026-09-16 — WiZink added

- **Scope widened to two sellers**, `$sellers = 'MILLENNIUM|WIZINK'`. Millennium is
  **completely unchanged**: same 11 batches, same 120,543 loans, same EUR 906,359,149.41,
  same EUR 160,045,164.05. Verified, not assumed — no loan names both a Millennium and a
  WiZink batch, so the wider scope cannot reallocate one.
- **Forward flow excluded**, `$deal_type = 'One-off'`. 129 of WiZink's 133 PAM batches are
  `Ongoing` — the monthly forward-flow deliveries. Without this predicate the benchmark
  silently becomes a forward-flow book.
- **±10% reconciliation on collections.** Of the four WiZink one-off purchases, only
  `PT18019A000000` lands within ±10% of PAM over the same window (104.50%, or 99.13% net of
  pre-capex cash). The other three read 67.2%, 153.1% and 39.2% and are excluded.
- **Interest dropped.** `REGISTRATION_INTEREST` and `REGISTRATION_BALANCE` are gone. Interest
  is delivered on 0.86% of Millennium loans and 93.2% of WiZink's, so a balance summed
  across the two adds two delivery standards together.
- **WiZink carried as pure consumer.** Its 5 company-debtor loans (EUR 30,332.57) are dropped
  in both files. Millennium keeps its 12,844 SME loans.
- **Debt age now computed here, not taken from PAM.** PAM's `AVERAGE_DEBT_AGE` is in *years*
  on Millennium and in *months* on WiZink — same column, same file, two units.
- **Pre-registration cash excluded**, `$drop_pre_registration = true` in `payments.sql`.
  Cash collected before Intrum opened the case is dropped: EUR 4,919,333.95 and 733 loans
  whose only cash predates registration. This makes CAT directly comparable to PAM, whose
  lifetime also starts at acquisition — every batch now reconciles between **97.82% and
  101.07%**, against 98.7%-105.7% before. The ±10% batch selection was re-checked on both
  bases and does not move. Set the variable to `false` to return to the old basis.


In [ ]:
import sys
sys.path.insert(0, "../../packages")   # notebook sits in work/project-elsa-benchmark/
import datalake
import pandas as pd

# Money and ratios to 5 dp, per the workspace convention. This changes only what
# is PRINTED — it never alters the values, so nothing is rounded away.
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")

# Name your export here. Leave it empty and the files are written as
# accounts.csv / payments.csv; set it and they become <name>_accounts.csv etc.
EXPORT_NAME = "project_elsa_benchmark"


In [ ]:
# ~30 seconds. Reads the CAT case feed and classifies borrowers from raw DEBTORS.
#
# convert=False keeps Snowflake's Decimal values instead of float64. It is the
# workspace rule for anything you SUM OR RECONCILE — and this is the valuation
# base. Trade-off: display.float_format does not apply to Decimal columns, so
# format money explicitly when you print it.
accounts_df = datalake.q("work/project-elsa-benchmark/accounts", convert=False)

# ⚠ convert=False also leaves DATE columns as Python date objects, and pandas
# cannot min/max/resample an object column. Coerce the dates back — this is
# safe, they carry no precision to lose. Only the money stays Decimal.
for col in ("REGISTRATION_DATE", "DEFAULT_DATE"):
    accounts_df[col] = pd.to_datetime(accounts_df[col])

print(f"{len(accounts_df):,} loans x {len(accounts_df.columns)} columns")
accounts_df.head()


In [ ]:
# ⚠ ~9 minutes. Reads every extract of a 24.7bn-row table with no date bound.
# It applies the same raw borrower classification as accounts.sql so the two
# populations match exactly — the assert below checks that they still do.
payments_df = datalake.q("work/project-elsa-benchmark/payments", convert=False)

# Same date coercion as above. Undated payments become NaT, which is what you
# want: they drop out of any time series while their money stays in the total.
payments_df["PAYMENT_DATE"] = pd.to_datetime(payments_df["PAYMENT_DATE"])

print(f"{len(payments_df):,} rows x {len(payments_df.columns)} columns")
payments_df.head()

In [ ]:
# Last payment per batch — the CAT extract cut, not the end of collections.
# Every batch stops at 2026-01-14 or earlier; PAM runs to 2026-07.
payments_df.groupby(["SELLER_NAME", "PORTFOLIO_ID"])["PAYMENT_DATE"].max()


In [ ]:
# --- sanity checks worth running before you trust either frame ---------------
# Each of these has failed at least once during development, which is why they
# are asserts and not comments.
assert accounts_df["LOAN_ID"].is_unique, "accounts must be one row per loan"
assert set(accounts_df["SELLER_NAME"]) == {"Millennium", "Wizink"}, \
    "both sellers must be present — check the $sellers / $deal_type scope in the .sql"
assert set(accounts_df["PRICING_SEGMENT"]) == {"Individual", "SME"}, \
    "pricing_segment should only ever be Individual or SME — Unknown is excluded in the .sql"
# WiZink is carried as a consumer book: its 5 company-debtor loans are dropped
# in BOTH .sql files. If this fails, one file was changed and the other was not.
assert not ((accounts_df["SELLER_NAME"] == "Wizink")
            & (accounts_df["PRICING_SEGMENT"] == "SME")).any(), \
    "WiZink must carry no SME loans — the E3 rule has drifted between the two files"
assert set(payments_df["LOAN_ID"]) <= set(accounts_df["LOAN_ID"]), \
    "payments must not contain a loan that accounts drops — the two exclusions have drifted apart"
# Nothing in either book is secured. Asserted rather than assumed so a future
# delivery cannot quietly introduce collateral into a book priced as unsecured.
assert (accounts_df["REGISTRATION_CAPITAL"] >= 0).all(), \
    "no negative capital existed when this was written"

total = payments_df["TOTAL_COLLECTIONS"].sum()
undated = payments_df.loc[payments_df["MISSING_PAYMENT_DATE_FLAG"], "TOTAL_COLLECTIONS"].sum()

print(f"batches              {accounts_df['PORTFOLIO_ID'].nunique():>12,}   (expect 12)")
print(f"loans                {len(accounts_df):>12,}   (expect 150,514)")
print(f"payment rows         {len(payments_df):>12,}")
print(f"collections total    {total:>12,.2f}")
print(f"  of which undated   {undated:>12,.2f}   ({undated/total:.2%} — no date, cannot be placed in a month)")
print(f"loans that ever paid {payments_df['LOAN_ID'].nunique():>12,}")
print()
# By seller. Formatted explicitly because display.float_format does not reach
# Decimal columns — a raw print shows 18 decimal places.
_cap = accounts_df["REGISTRATION_CAPITAL"].sum()
print(f"{'':12}{'loans':>9}{'capital':>18}{'% of capital':>14}")
for _s, _g in accounts_df.groupby("SELLER_NAME"):
    _c = _g["REGISTRATION_CAPITAL"].sum()
    print(f"{_s:12}{len(_g):>9,}{_c:>18,.2f}{_c / _cap * 100:>13.2f}%")
print(f"{'TOTAL':12}{len(accounts_df):>9,}{_cap:>18,.2f}{100:>13.2f}%")
print()
print(f"{'':12}{'loans':>9}{'capital':>18}")
for (_s, _seg), _g in accounts_df.groupby(["SELLER_NAME", "PRICING_SEGMENT"]):
    print(f"{_s + ' ' + _seg:24}{len(_g):>9,}{_g['REGISTRATION_CAPITAL'].sum():>18,.2f}")

# ⚠ REGISTRATION_CAPITAL is the ONLY money column in accounts, by decision.
# Interest was dropped 2026-09-16: it is delivered on 0.86% of Millennium loans
# and 93.2% of WiZink's, so a balance summed across the two books would add two
# different delivery standards together. Capital is complete on every loan in
# both, and is the figure that reconciles to PAM.
#
# Filter on PAYMENT_DATE for any time series — but expect the total to fall
# short by the undated figure above. Do not filter them out and then quote the
# remainder as the book's collections.


## ⚠ Read before exporting — everything worth noting, in one place

### The numbers this run should produce

| | loans | EUR |
|---|---|---|
| `accounts` loans | 150,514 | |
| &nbsp;&nbsp;Millennium | 120,543 | 906,359,149.41 |
| &nbsp;&nbsp;WiZink | 29,971 | 122,891,559.16 |
| &nbsp;&nbsp;**capital, total** | | **1,029,250,708.57** |
| &nbsp;&nbsp;SME — Millennium only | 12,844 | 445,511,420.06 |
| &nbsp;&nbsp;Individual | 137,670 | 583,739,288.51 |
| `payments` rows / paying loans | 798,774 / 67,503 | 201,224,598.53 |
| &nbsp;&nbsp;of which **undated** (2.69%, all Millennium) | | 5,407,533.41 |

Lifetime recovery on registration capital: Millennium **17.442%** · WiZink **35.109%** ·
combined 19.551%. **The combined figure describes neither book.**

Average debt age (registration − default, Intrum's own definition):
Millennium 4.27276 years · WiZink 2.82206 years.

### Say these five things whenever you quote a figure

1. **Which seller.** The two books are 13 and 8 years apart in vintage, were bought at ~11%
   and ~20% of face, and recover at 17.442% and 35.109%. Almost nothing is safe to blend.
2. **Capital, not balance.** Interest no longer exists in this dataset, deliberately —
   it is delivered on 0.86% of Millennium loans and 93.2% of WiZink's.
3. **`SME` means "a corporate debtor is on the loan"** — not a size class, and **Millennium
   only**. 11,374 of the 12,844 are SME by ratified policy, not evidence.
4. **Which recovery denominator** — `PT050030000000` alone reads **40.44% or 50.97%**
   depending on the choice.
5. **This is not "collections to date."** CAT stops at its extract, 2026-01-14; PAM runs to
   2026-07. Read anything recent from PAM.

### Never

- Filter out the undated cash and then quote the remainder as the book's collections.
- Blend a recovery rate, a debt age or a segment split across the two sellers without saying so.
- Join `accounts` to `payments` without aggregating payments to loan level **first** — and
  use a LEFT join, or you drop the loans that never paid.
- Quote a promissory-note or instrument rate that includes WiZink in the denominator. Its
  `CLAIM_REASON` field contains **dates**, not instrument names — the question cannot be
  asked of that book at all, which is not the same as the answer being zero.
- Quote PAM's `AVERAGE_DEBT_AGE`. Years on Millennium, months on WiZink, 0.00 on
  `PT15018G000000` and 31 on `PT18019A000000` against 6.39 and 2.84 measured.
- Quote a CAT-vs-PAM ratio without stating the period cut.
- Put `SME` into anything regulatory or capital-related.
- Reuse either file outside PT / Millennium + WiZink without re-reading `payments.sql` N5.


In [ ]:
# Convenience reload, to skip the ~4.5-minute payments query in a fresh kernel.
# ⚠ This reads the LAST EXPORT, not the warehouse. If the .sql files have
# changed since, you are loading stale data — rerun cells 2-3 instead.
# Skipped silently when the export does not exist yet, so a top-to-bottom run
# of a fresh notebook does not fail here.
import os
_stem = ("/Users/sharlenechin/data-warehouse/local-workspace/work/"
         "project-elsa-benchmark/project_elsa_benchmark")
if os.path.exists(f"{_stem}_accounts.csv") and os.path.exists(f"{_stem}_payments.csv"):
    accounts_df = pd.read_csv(f"{_stem}_accounts.csv",
                              parse_dates=["REGISTRATION_DATE", "DEFAULT_DATE"])
    payments_df = pd.read_csv(f"{_stem}_payments.csv", parse_dates=["PAYMENT_DATE"])
    print(f"reloaded from CSV: {len(accounts_df):,} loans, {len(payments_df):,} payment rows")
else:
    print("no export on disk yet — keeping the frames loaded from the warehouse")


In [ ]:
# --- portfolio summary, one row per batch -------------------------------------
# The acquisition-side table: what Intrum says it bought, beside what we hold,
# with a line on every batch saying why it is in or out.
#
# ⚠ THE ACQUISITION COLUMNS COME FROM PAM, NOT FROM accounts_df. Capex, price
# and acquired balance are deal-level facts only PAM carries; a roll-up of our
# loan-level data is a different quantity and is shown separately below.
pam_df = datalake.q("work/project-elsa-benchmark/portfolio_summary", convert=False)

# ⚠ BOTH DATE COLUMNS MUST BE DATETIMES BEFORE THE RULES BELOW RUN. Cells 2-3
# coerce them after datalake.q, but reloading the exported CSV with a plain
# pd.read_csv (no parse_dates=) brings them back as STRINGS, and then:
#   · .dt.year raises AttributeError, and
#   · DEFAULT_DATE > REGISTRATION_DATE silently becomes a STRING comparison,
#     which happens to agree for ISO dates and would not for any other format.
# Coercing here costs nothing and makes the cell correct from either source.
for _col in ("REGISTRATION_DATE", "DEFAULT_DATE"):
    accounts_df[_col] = pd.to_datetime(accounts_df[_col])

# The six batches accounts.sql and payments.sql drop, and why. Kept as text
# here so the summary can state the reason rather than leave a blank row.
# The three WiZink ones were added 2026-09-16: PAM's forward-flow batches are
# excluded wholesale by $deal_type, and of the four WiZink one-off purchases
# only PT18019A000000 reconciles to PAM on collections within ±10%.
EXCLUDED_BATCH_REASONS = {
    "PT06014B010000": "EXCLUDED, whole batch — no cases in CAT under any source system; a feed "
                      "gap, not an empty book. Its single PAM month equals its own acquisition "
                      "principal to the cent: an accounting entry, not cash.",
    "PT070040000000": "EXCLUDED, whole batch — 2 loans holding EUR 2.00 of capital each, "
                      "registered 279 days after capex where every other batch lands within -2 "
                      "to +40. Never paid. Its single PAM month is the same accounting entry.",
    "PT15018A040000": "EXCLUDED, whole batch — its 84 cases resolve to PT16023C000000 under "
                      "latest-in-scope-wins, so it was never in the output. PAM's acquired "
                      "principal has no loans behind it; second-largest open Intrum item.",
    "PT19007D010000": "EXCLUDED, whole batch — CAT holds 67.2% of the collections PAM reports "
                      "for it over the same window, far outside the ±10% the other batches "
                      "meet. Only 3,604 loans found against PAM's 4,633 accounts, and "
                      "AGE_OF_DEBT is delivered on 7.2% of them. Thinly delivered.",
    "PT19013A020000": "EXCLUDED, whole batch — CAT holds 153.1% of PAM's collections over the "
                      "same window. Over-delivery this large is not pre-capex cash; it is "
                      "unexplained and one for Intrum.",
    "PT19013B010000": "EXCLUDED, whole batch — CAT holds 39.2% of PAM's collections over the "
                      "same window. 232 loans against PAM's 310 accounts.",
}

# What stages/01-cleaning drops downstream of THIS export, recomputed from the
# same rules rather than quoted, so the counts cannot drift from the code.
#   zero_capital    registration_capital <= 0     ⚠ WAS registration_balance
#                                                   until 2026-09-16; interest
#                                                   no longer exists, and the
#                                                   rule catches 16 loans on
#                                                   capital against 7 on balance
#   not_in_default  default_date > registration_date
#   one-account vintage             a registration year holding exactly ONE
#                                   loan. Millennium 2014 (EUR 2,074.48) and,
#                                   added 2026-09-16, WiZink 2019 (loan
#                                   13634876, EUR 501.54). Both are healthy
#                                   loans dropped only because a vintage of one
#                                   cannot carry a curve. The years are named
#                                   PER SELLER rather than detected, so the
#                                   rule cannot silently start dropping a real
#                                   vintage that happens to thin out.
ONE_ACCOUNT_VINTAGES = {"Millennium": [2014], "Wizink": [2019]}

EXCLUSION_RULES = {
    "zero capital":        lambda df_: df_["REGISTRATION_CAPITAL"] <= 0,
    "not in default":      lambda df_: df_["DEFAULT_DATE"] > df_["REGISTRATION_DATE"],
    "one-account vintage": lambda df_: pd.concat(
        [(df_["SELLER_NAME"] == _s) & (df_["REGISTRATION_DATE"].dt.year.isin(_y))
         for _s, _y in ONE_ACCOUNT_VINTAGES.items()], axis=1).any(axis=1),
}

portfolio_exclusions_df = (
    pd.concat(
        [accounts_df.loc[test(accounts_df)]
                    .groupby(["SELLER_NAME", "PORTFOLIO_ID"])
                    .agg(accounts=("LOAN_ID", "size"),
                         capital=("REGISTRATION_CAPITAL", "sum"))
                    .assign(rule=name)
         for name, test in EXCLUSION_RULES.items()]
    )
    .reset_index()
    .rename(columns={"SELLER_NAME": "Seller", "PORTFOLIO_ID": "Intrum Portfolio ID",
                     "rule": "Exclusion Rule", "accounts": "Accounts", "capital": "Capital"})
    .sort_values(["Intrum Portfolio ID", "Exclusion Rule"], ignore_index=True)
    .astype({"Accounts": "int64", "Capital": "float64"})
)

_dropped = pd.concat([test(accounts_df) for test in EXCLUSION_RULES.values()], axis=1).any(axis=1)

# ⚠ DEBT AGE IS COMPUTED HERE, NOT TAKEN FROM PAM (changed 2026-09-16).
# PAM's AVERAGE_DEBT_AGE is in YEARS on the Millennium batches and in MONTHS on
# the WiZink ones — same column, same file, two units, with nothing in the data
# saying which is which. It also reported 0.00 on PT15018G000000 and 31 on
# PT18019A000000, against 6.39 and 2.84 years measured. This calculation
# reproduces Intrum's own per-case AGE_OF_DEBT field (which is in DAYS) to five
# decimal places, so it is their definition, not ours.
ours = accounts_df.assign(
    _dropped=_dropped,
    _debt_age_years=(accounts_df["REGISTRATION_DATE"]
                     - accounts_df["DEFAULT_DATE"]).dt.days / 365.25,
).groupby(["SELLER_NAME", "PORTFOLIO_ID"]).agg(
    our_loans=("LOAN_ID", "size"),
    our_capital=("REGISTRATION_CAPITAL", "sum"),       # Decimal in, exact sum
    our_debt_age_years=("_debt_age_years", "mean"),
    accounts_excluded=("_dropped", "sum"),
)
ours["our_collections"] = payments_df.groupby(["SELLER_NAME", "PORTFOLIO_ID"])["TOTAL_COLLECTIONS"].sum()
ours["our_loans_after_cleaning"] = ours["our_loans"] - ours["accounts_excluded"]

# LEFT from PAM, so every in-scope batch appears and the six we drop show as
# unmatched rather than silently vanishing — that absence is the point.
portfolio_detail_df = (pam_df.set_index(["SELLER_NAME", "PORTFOLIO_ID"])
                             .join(ours).reset_index())

# Decimal only mattered through the summation; float from here is for display
# and for the ratios. Money is never rounded, only formatted.
for _col in ("CAPEX_AMOUNT", "ACQ_BALANCE", "PRICE_PCT", "PAM_COLLECTIONS",
             "PAM_COLLECTIONS_TO_CAT_CUT", "our_capital", "our_collections"):
    portfolio_detail_df[_col] = portfolio_detail_df[_col].astype(float)

# The reconciliation the ±10% rule is applied on. BOTH SIDES ARE CUT AT THE CAT
# EXTRACT (2026-01) — PAM runs to 2026-07 and a whole-against-whole ratio is not
# like-for-like. See payments.sql READ FIRST 2.
portfolio_detail_df["cat_pct_of_pam"] = (
    portfolio_detail_df["our_collections"]
    / portfolio_detail_df["PAM_COLLECTIONS_TO_CAT_CUT"] * 100)

# ⚠ COUNTS MUST STAY COUNTS. The LEFT join puts NaN on the excluded batches,
# and NaN is a float, so every integer column silently promotes — "406 accounts
# excluded" renders as 406.00000. Int64 is pandas' NULLABLE integer.
for _col in ("CAPEX_YEAR", "ACQ_ACCOUNTS", "ACQ_DEBTORS", "our_loans",
             "accounts_excluded", "our_loans_after_cleaning"):
    portfolio_detail_df[_col] = portfolio_detail_df[_col].astype("Int64")

_drops_by_batch = (
    portfolio_exclusions_df
    .assign(text=lambda df_: df_["Accounts"].astype(int).astype(str) + " " + df_["Exclusion Rule"])
    .groupby("Intrum Portfolio ID")["text"].apply("; ".join)
)


def _commentary(row):
    """One line per batch: in or out, and exactly what cleaning removes from it."""
    batch = row["Intrum Portfolio ID"]
    if batch in EXCLUDED_BATCH_REASONS:
        return EXCLUDED_BATCH_REASONS[batch]
    drops = _drops_by_batch.get(batch)
    parts = [f"Included — {int(row['our_loans']):,} loans exported, "
             f"EUR {row['our_capital']:,.2f} registration capital."]
    parts.append(f"Cleaning then drops {drops}, leaving "
                 f"{int(row['our_loans_after_cleaning']):,}."  if drops
                 else "Cleaning drops nothing; all loans survive.")
    parts.append(f"Collections reconcile at {row['cat_pct_of_pam']:,.1f}% of PAM "
                 f"over the same window.")
    return " ".join(parts)


portfolio_detail_df = portfolio_detail_df.rename(columns={
    "SELLER_NAME":     "Seller",
    "PORTFOLIO_ID":    "Intrum Portfolio ID",
    "CAPEX_YEAR":      "Capex Year",
    "CAPEX_AMOUNT":    "Capex Amount",
    "PRICE_PCT":       "Price %",
    "ACQ_BALANCE":     "Acq Balance",
    "ACQ_ACCOUNTS":    "Acq Accounts",
    "PAM_COLLECTIONS": "Collections",
})
portfolio_detail_df["Avg. Debt Age (Years)"] = portfolio_detail_df["our_debt_age_years"]
portfolio_detail_df["Accounts Excluded"] = portfolio_detail_df["accounts_excluded"]
portfolio_detail_df["Commentary / Exclusions"] = portfolio_detail_df.apply(_commentary, axis=1)

portfolio_summary_df = portfolio_detail_df[[
    "Seller", "Intrum Portfolio ID", "Capex Year", "Capex Amount", "Price %", "Acq Balance",
    "Acq Accounts", "Avg. Debt Age (Years)", "Collections", "Accounts Excluded",
    "Commentary / Exclusions",
]]

# ⚠ Acq Accounts is PAM's ACQCASES, counted at acquisition. It is NOT our loan
# count and the two must never be differenced — PT050030000000 reads 42,011
# here against 28,444 loans in accounts_df. Both are in portfolio_detail_df.
# ⚠ Collections is PAM's LIFETIME figure, running to 2026-08. Ours stops at the
# CAT extract, 2026-01-14, and is in portfolio_detail_df as `our_collections`.
# The reconciliation column uses PAM cut at the same date, not this one.
_excluded = int(portfolio_exclusions_df["Accounts"].sum())
print(f"{len(portfolio_summary_df)} PAM batches in scope, "
      f"{int(portfolio_detail_df['our_loans'].notna().sum())} of them in our datasets")
print(f"\nexcluded accounts, by rule:")
print(portfolio_exclusions_df.to_string(index=False))
print(f"\n{len(accounts_df):,} exported - {_excluded} = "
      f"{len(accounts_df) - _excluded:,} after cleaning")
portfolio_summary_df


In [ ]:
# --- export ------------------------------------------------------------------
stem = f"{EXPORT_NAME}_" if EXPORT_NAME else ""
for name, df in (("accounts", accounts_df), ("payments", payments_df)):
    path = f"{stem}{name}.csv"
    df.to_csv(path, index=False)
    print(f"{len(df):>9,} rows -> {path}")